# Transfer pixel from a photo to recreate another photo

## Idea:
- just one fct to do it to get the new position in same frame,
then we can decide either to transfer pixel in same frame or to another one:
e.g. transfer pixel from frame on the left to frame on the right, or from upper frame to downside frame
=> we can place 4 famous paintings in a big canvas, then transfer pixels from each painting to their neighbor clockwise.

## Import modules

In [1]:
# import internal modules
from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date
from dataclasses import dataclass, field

# import 3rd-party modules
import cv2
import numpy as np
from numba import njit
from pygifsicle import optimize


# import local modules
from utils.renderer.giffer import create_gif
from utils.renderer.videographer import create_video
from utils.project_manager import Project
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [1]:
# check current working directory in notebook session
!pwd

/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core


In [2]:
# name out img dirs
out_img_dir_list = ["transfer_same_canvas", "one_transfer", "quadruple_transfer"]

# create project
project = Project(project_dir="assets/images/pixel_transfers", in_img_dir="db", out_img_dir_list=out_img_dir_list)

## Define functions and classes

In [3]:
# decorate function with numba fct to speed up execution
@njit()
def get_pixel_coords_map(src_img, dest_img):
    """
    Function to find best matching pixel for region of interest (a pixel) of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with pixel from source image
    """
    # create empty lists (or arrays) to store coords mapping between source and dest images
    src_yxs = []
    dest_yxs = []

    # get list of all pixel coords in src and dest images
    dest_pixel_coords = [(dest_y, dest_x) for dest_y in range(dest_img.shape[0]) for dest_x in range(dest_img.shape[1])]
    src_pixel_coords = [(src_y, src_x) for src_y in range(src_img.shape[0]) for src_x in range(src_img.shape[1])]

    # shuffle dest_pixel_coords (otherwise the first pixels from top will get the best matching pixels)
    dest_pixel_idxs = np.arange(len(dest_pixel_coords))
    np.random.seed(1111)
    np.random.shuffle(dest_pixel_idxs)

    # iterate over each pixel position
    for i in dest_pixel_idxs:

        dest_y, dest_x = dest_pixel_coords[i]
        # get region of interest (pixel) in destination img
        dest_pixel = dest_img[dest_y, dest_x]


        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each pixel in source image
        for src_pixel_idx, (src_y, src_x) in enumerate(src_pixel_coords):
            # get source pixel
            src_pixel = src_img[src_y, src_x]

            # compute distance between the pixels
            dist = np.sum(np.abs(dest_pixel - src_pixel))
            
            # if distance is smaller than the current best dist
            if dist < best_match_dist:
                # update current best dist
                best_match_dist = dist
                # store best match index 
                best_match_index = src_pixel_idx

        # take out best src pixel coords from list of src pixel coords
        (best_match_y, best_match_x) = src_pixel_coords.pop(best_match_index)

        # append best src pixel coords and their corresponding dest pixel coords to lists
        src_yxs.append((best_match_y, best_match_x))
        dest_yxs.append((dest_y, dest_x))

    return src_yxs, dest_yxs

In [4]:
# create class for mapping pixel from source image to dest image
@dataclass
class PixelMap:
    src_yx: tuple
    dest_yx: tuple
    steps: int
    X: np.array = field(init=False, repr=True)
    Y: np.array = field(init=False, repr=True)

    def __post_init__(self):
        """
        Special method called just after the init to initialize internal attributes
        that depends on the previous attributes.

        Thanks to field(), you can still instantiate the class without setting X and Y
        """
        # get line range from start yx coords and end yx coords
        self.X = np.linspace(self.src_yx[1], self.dest_yx[1], self.steps, dtype=np.uint64)
        self.Y = np.array = np.linspace(self.src_yx[0], self.dest_yx[0], self.steps, dtype=np.uint64)

## Get images paths

In [7]:
# # set img_path_list
# project.img_path_list = [
#     "/assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg",
#     "/assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg",
#     "core/assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg",
#     "core/assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg"
# ]

In [8]:
# get img_path_list
project.in_img_path_list

['assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
 'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
 'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
 'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg']

In [5]:
# read first image to get a reference shape
ref_img = cv2.imread(project.in_img_path_list[0])
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape

In [8]:
nb_imgs = len(project.in_img_path_list)

# get src-dest pair of img paths for mapping
src_dest_pairs = []
for img_idx in range(nb_imgs):
    src_dest_pairs.append((project.in_img_path_list[img_idx], project.in_img_path_list[(img_idx + 1)%nb_imgs]))
src_dest_pairs

[('assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
  'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg'),
 ('assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
  'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg'),
 ('assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
  'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg'),
 ('assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg',
  'assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg')]

## Read images 
- resized to same shape as reference image
- and with padding or cropped to keep same aspect ratio

In [11]:
for src_img_path, dest_img_path in src_dest_pairs:
    src_img = resize_with_crop(img_path=src_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel), cvt_color=cv2.COLOR_BGR2LAB)
    dest_img = resize_with_crop(img_path=dest_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel), cvt_color=cv2.COLOR_BGR2LAB)

    # show alongside reference image and dest image with padding and resized
    cv2.imshow("result",cv2.hconcat([src_img, dest_img]))

    # wait for any press on keyboard
    cv2.waitKey(0)

    # destroy all windows
    cv2.destroyAllWindows()
    cv2.waitKey(1) # workaround on mac to effectively close the windows

img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641


## Get yx coords mapping from one photo to another

In [6]:
# create directory where to save yx coords mapping
project.make_dir("pixel_map")
project.out_img_dir_dict

{'transfer_same_canvas': PosixPath('assets/images/pixel_transfers/transfer_same_canvas'),
 'one_transfer': PosixPath('assets/images/pixel_transfers/one_transfer'),
 'quadruple_transfer': PosixPath('assets/images/pixel_transfers/quadruple_transfer'),
 'pixel_map': PosixPath('assets/images/pixel_transfers/pixel_map')}

In [9]:
# iterate over each images pair
for i, (src_img_path, dest_img_path) in enumerate(src_dest_pairs):
    # resize while keeping same aspect ratio (thanks to padding or cropping)
    src_img = resize_with_crop(img_path=src_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel), cvt_color=cv2.COLOR_BGR2LAB)
    dest_img = resize_with_crop(img_path=dest_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel), cvt_color=cv2.COLOR_BGR2LAB)

    # get yx coords mapping from src to dest
    src_yxs, dest_yxs = get_pixel_coords_map(src_img, dest_img)

    # # save yx coords mapping as numpy array of 4 cols: src_y, src_x, dest_y, dest_x
    # pixel_map_np = np.column_stack((src_yxs, dest_yxs))
    # np.save(project.out_img_dir_dict["pixel_map"] / f"{i}.npy", pixel_map_np)

img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641


## Create images to visualize pixel transfer on same canvas

In [39]:
# choose index of src-dest pair
src_dest_idx = 3

# set current out image directory
out_img_dir = f"transfer_same_canvas_{src_dest_idx}"

# create directory where to images
project.make_dir(out_img_dir)
project.out_img_dir_dict

{'transfer_same_canvas': PosixPath('assets/images/pixel_transfers/transfer_same_canvas'),
 'one_transfer': PosixPath('assets/images/pixel_transfers/one_transfer'),
 'quadruple_transfer': PosixPath('assets/images/pixel_transfers/quadruple_transfer'),
 'pixel_map': PosixPath('assets/images/pixel_transfers/pixel_map'),
 'one_transfer2': PosixPath('assets/images/pixel_transfers/one_transfer2'),
 'one_transfer_1': PosixPath('assets/images/pixel_transfers/one_transfer_1'),
 'one_transfer_0': PosixPath('assets/images/pixel_transfers/one_transfer_0'),
 'one_transfer_4': PosixPath('assets/images/pixel_transfers/one_transfer_4'),
 'one_transfer_3': PosixPath('assets/images/pixel_transfers/one_transfer_3'),
 'transfer_same_canvas_3': PosixPath('assets/images/pixel_transfers/transfer_same_canvas_3')}

In [64]:
# get list of pixels mapping
pixel_map_paths = list(project.out_img_dir_dict["pixel_map"].glob('*.npy'))
pixel_map_paths = sorted(pixel_map_paths)

# load pixels mapping
pixel_map_np = np.load(pixel_map_paths[src_dest_idx])

# set pixel speed (every pixel should arrive to dest yx after 60 frames)
steps = 60

# get list of PixelMap objects
pixel_map_list = [PixelMap(src_yx=(pixel_map[0],pixel_map[1]), dest_yx=(pixel_map[2],pixel_map[3]), steps=steps) for pixel_map in pixel_map_np]

In [60]:
pixel_map_list = [PixelMap(src_yxs[i], dest_yxs[i], steps) for i in range(len(src_yxs))]

In [65]:
src_img_path = src_dest_pairs[src_dest_idx][0]
# resize while keeping same aspect ratio (thanks to padding or cropping)
src_img = resize_with_crop(img_path=src_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))

img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641


In [37]:
src_dest_pairs[src_dest_idx]

('assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg',
 'assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg')

In [66]:
# copy source image
out_img = src_img.copy()

# move pixels from source position to destination position `steps`` times
for i in range(steps):

    # iterate over each pixel map
    for pixel_map in pixel_map_list:

        # get src pixel coords
        src_y, src_x = pixel_map.src_yx

        # get src pixel
        src_pixel = src_img[src_y, src_x, :]

        # get new coords for src pixel
        new_x = pixel_map.X[i]
        new_y = pixel_map.Y[i]

        # update image with src pixel at new coords
        out_img[new_y, new_x, :] = src_pixel

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

## Create gif or video from output images

In [5]:
# get current date
today = date.today().strftime("%Y%m%d")

# set gif path
gif_path = str(project.project_dir / f"{out_img_dir}_{today}.gif")

# set output shape
out_img_height, out_img_width = ref_img_height//1, ref_img_width//1

# create gif
create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(out_img_height, out_img_width), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5)

# optimize gif to reduce size
optimize(gif_path)

In [ ]:
# get current date
today = date.today().strftime("%Y%m%d")

# set gif path
gif_path = str(project.project_dir / f"{out_img_dir}_{today}.gif")

# set output shape
out_img_height, out_img_width = ref_img_height//5, ref_img_width//5

create_video(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(out_img_height, out_img_width), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5)

## Create images to visualize pixel transfer on horizontal canvas

In [42]:
# choose index of src-dest pair
src_dest_idx = 3

# set current out image directory
out_img_dir = f"one_transfer_{src_dest_idx}"

# create directory where to images
project.make_dir(out_img_dir)
project.out_img_dir_dict

{'transfer_same_canvas': PosixPath('assets/images/pixel_transfers/transfer_same_canvas'),
 'one_transfer': PosixPath('assets/images/pixel_transfers/one_transfer'),
 'quadruple_transfer': PosixPath('assets/images/pixel_transfers/quadruple_transfer'),
 'pixel_map': PosixPath('assets/images/pixel_transfers/pixel_map'),
 'one_transfer2': PosixPath('assets/images/pixel_transfers/one_transfer2'),
 'one_transfer_1': PosixPath('assets/images/pixel_transfers/one_transfer_1'),
 'one_transfer_0': PosixPath('assets/images/pixel_transfers/one_transfer_0'),
 'one_transfer_4': PosixPath('assets/images/pixel_transfers/one_transfer_4'),
 'one_transfer_3': PosixPath('assets/images/pixel_transfers/one_transfer_3'),
 'transfer_same_canvas_3': PosixPath('assets/images/pixel_transfers/transfer_same_canvas_3')}

In [67]:
# get list of pixels mapping
pixel_map_paths = list(project.out_img_dir_dict["pixel_map"].glob('*.npy'))
pixel_map_paths = sorted(pixel_map_paths)

# load pixels mapping
pixel_map_np = np.load(pixel_map_paths[src_dest_idx])

# set pixel speed (every pixel should arrive to dest yx after 60 frames)
steps = 60

# set offset yx to transfer pixel to the right part of canvas
offset_x = ref_img_width
offset_y = 0

# get list of PixelMap objects
pixel_map_list = [PixelMap(src_yx=(pixel_map[0],pixel_map[1]), dest_yx=(pixel_map[2] + offset_y,pixel_map[3] + offset_x), steps=steps) for pixel_map in pixel_map_np]

In [68]:
pixel_map_paths

[PosixPath('assets/images/pixel_transfers/pixel_map/0.npy'),
 PosixPath('assets/images/pixel_transfers/pixel_map/1.npy'),
 PosixPath('assets/images/pixel_transfers/pixel_map/2.npy'),
 PosixPath('assets/images/pixel_transfers/pixel_map/3.npy')]

In [44]:
src_img_path = src_dest_pairs[src_dest_idx][0]
# resize while keeping same aspect ratio (thanks to padding or cropping)
src_img = resize_with_crop(img_path=src_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))

img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641


In [69]:
# set canvas
black_canvas = np.zeros_like(ref_img)
out_img = cv2.hconcat([src_img, black_canvas])


# get range to set images blending alpha at each step
alpha_range = np.linspace(0, 1, steps)

# move pixels from source position to destination position `steps`` times
for i in range(steps):

    # iterate over each pixel map
    for pixel_map in pixel_map_list:

        # get src pixel coords
        src_y, src_x = pixel_map.src_yx

        # get src pixel
        src_pixel = src_img[src_y, src_x, :]

        # get new coords for src pixel
        new_x = pixel_map.X[i]
        new_y = pixel_map.Y[i]

        # update image with src pixel at new coords
        out_img[new_y, new_x, :] = src_pixel

        if i == 1:
            # black out src pixel at src coords
            out_img[src_y, src_x, :] = 0


    # blend images: make first half slowly turning to black
    alpha = alpha_range[i]
    beta = 1 - alpha
    out_img[:,:out_img.shape[1]//2,:] = cv2.addWeighted(black_canvas,alpha, out_img[:,:out_img.shape[1]//2,:],beta, 0)

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

In [27]:
# set canvas
black_canvas = np.zeros_like(ref_img)
out_img = cv2.hconcat([src_img, black_canvas])

# show alongside reference image and dest image with padding and resized
cv2.imshow("result",out_img)

# wait for any press on keyboard
cv2.waitKey(0)

# destroy all windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround on mac to effectively close the windows

-1

## Create images to visualize 4 pixel transfers

In [74]:
# set current out image directory
out_img_dir = f"quadruple_transfer"

# create directory where to images
project.make_dir(out_img_dir)
project.out_img_dir_dict

{'transfer_same_canvas': PosixPath('assets/images/pixel_transfers/transfer_same_canvas'),
 'one_transfer': PosixPath('assets/images/pixel_transfers/one_transfer'),
 'quadruple_transfer': PosixPath('assets/images/pixel_transfers/quadruple_transfer'),
 'pixel_map': PosixPath('assets/images/pixel_transfers/pixel_map'),
 'one_transfer2': PosixPath('assets/images/pixel_transfers/one_transfer2'),
 'one_transfer_1': PosixPath('assets/images/pixel_transfers/one_transfer_1'),
 'one_transfer_0': PosixPath('assets/images/pixel_transfers/one_transfer_0'),
 'one_transfer_4': PosixPath('assets/images/pixel_transfers/one_transfer_4'),
 'one_transfer_3': PosixPath('assets/images/pixel_transfers/one_transfer_3'),
 'transfer_same_canvas_3': PosixPath('assets/images/pixel_transfers/transfer_same_canvas_3')}

In [72]:
pixel_map_lists = []
offset_list = [
    (0,0,0,ref_img_width),
    (0,ref_img_width,ref_img_height,ref_img_width),
    (ref_img_height,ref_img_width,ref_img_height,0),
    (ref_img_height,0, 0,0)
    ]

# set pixel speed (every pixel should arrive to dest yx after 60 frames)
steps = 60

# get list of pixels mapping
pixel_map_paths = list(project.out_img_dir_dict["pixel_map"].glob('*.npy'))
pixel_map_paths = sorted(pixel_map_paths)

# iterate over each pixel map path
for i, pixel_map_path in enumerate(pixel_map_paths):
    # load pixels mapping
    pixel_map_np = np.load(pixel_map_path)

    # get list of PixelMap objects
    pixel_map_list = [
        PixelMap(
            src_yx=(pixel_map[0] + offset_list[i][0], pixel_map[1] + offset_list[i][1]),
            dest_yx=(pixel_map[2] + + offset_list[i][2],pixel_map[3] + offset_list[i][3]),
            steps=steps) for pixel_map in pixel_map_np]
    pixel_map_lists.append(pixel_map_list)



In [75]:
img_list = []
for img_path in project.in_img_path_list:
    # resize while keeping same aspect ratio (thanks to padding or cropping)
    img = resize_with_crop(img_path=img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))

    img_list.append(img)

img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641


In [81]:
project.in_img_path_list

['assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
 'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
 'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
 'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg']

In [82]:
src_dest_pairs

[('assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
  'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg'),
 ('assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
  'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg'),
 ('assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
  'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg'),
 ('assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg',
  'assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg')]

In [114]:
# set canvas
out_img = cv2.vconcat([cv2.hconcat([img_list[0], img_list[1]]), cv2.hconcat([img_list[3], img_list[2]])])
src_canvas = out_img.copy()
black_canvas = np.zeros_like(src_canvas)

# get range to set images blending alpha at each step
alpha_range = np.linspace(0, 1, steps)

# move pixels from source position to destination position `steps`` times
for i in range(steps):

    # iterate over each pixel map
    for pixel_maps in pixel_map_lists:

        for pixel_map in pixel_maps:

            # get src pixel coords
            src_y, src_x = pixel_map.src_yx

            # get src pixel
            src_pixel = src_canvas[src_y, src_x, :]

            # get new coords for src pixel
            new_x = pixel_map.X[i]
            new_y = pixel_map.Y[i]

            if 0 < i < steps-1:
                # black out src pixel at src coords
                out_img[pixel_map.Y[i-1], pixel_map.X[i-1], :] = 0

            # update image with src pixel at new coords
            out_img[new_y, new_x, :] = src_pixel

            # elif i%6 == 0:
            #     # black out src pixel at src coords
            #     out_img[new_y, new_x, :] = 0


    # # blend images: make first half slowly turning to black
    # alpha = alpha_range[i]
    # beta = 1 - alpha
    # out_img[:,:out_img.shape[1]//2,:] = cv2.addWeighted(black_canvas[:,:out_img.shape[1]//2,:],alpha, out_img[:,:out_img.shape[1]//2,:],beta, 0)

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

In [115]:
# get current date
today = date.today().strftime("%Y%m%d")

# set gif path
gif_path = str(project.project_dir / f"{out_img_dir}_{today}.gif")

# set output shape
out_img_height, out_img_width = ref_img_height//1, ref_img_width//1

# create gif
create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(out_img_height, out_img_width), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5)

# optimize gif to reduce size
optimize(gif_path)

In [1]:
# import internal modules
from pathlib import Path

# import 3rd-party modules
import cv2
from utils.renderer.giffer import create_gif

In [12]:
from pygifsicle import optimize

In [14]:
# set output shape
out_img_height, out_img_width = 374, 266

# set img_dir
img_dir = "assets/images/pixel_transfers/quadruple_transfer"

# set gif path
gif_path = "assets/images/pixel_transfers/quadruple_transfer_20220318.gif"

# create gif
create_gif(img_dir=img_dir, out_path=gif_path, out_img_shape=(out_img_height, out_img_width), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5)

# optimize gif to reduce size
optimize(gif_path)

In [4]:
# set video path
video_path = str(project.project_dir / f"{out_img_dir}_{today}.mp4")

# create video
create_video(img_dir=project.out_img_dir_dict[out_img_dir], out_path=video_path, out_img_shape=(out_img_height, out_img_width), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5)

In [118]:
print(project.out_img_dir_dict[out_img_dir])
print(video_path)
print(out_img_height, out_img_width)

assets/images/pixel_transfers/quadruple_transfer
assets/images/pixel_transfers/quadruple_transfer_20220318.mp4
374 266


In [ ]:
# set video path
# video_path = str(project.project_dir / f"{out_img_dir}_{today}.mp4")

# create video
create_video(img_dir="assets/images/pixel_transfers/quadruple_transfer", out_path="assets/images/pixel_transfers/quadruple_transfer_20220318.mp4", out_img_shape=(374, 266), 
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=5, fps=20)